# 1. Configuracion base


In [3]:
from pyspark.sql import SparkSession
import time
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql.functions import col
spark = SparkSession.builder \
    .appName("spark_labs") \
    .master("local[6]") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.memory", "4g") \
    .config("spark.default.parallelism", "12") \
    .config("spark.sql.shuffle.partitions", "12") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()
print(spark.sparkContext.uiWebUrl)


http://host.docker.internal:4041


# 2. Partición Fisica
* Aquí sí se mueve data

In [5]:
df = spark.read.csv("./src/ventas.csv", header=True, inferSchema=True)

df_filtrado = df.filter(col("monto") > 100)

df_group1 = df_filtrado.groupBy("pais").sum("monto")
df_group2 = df_filtrado.groupBy("producto").sum("monto")


df_filter = df.filter(df.pais == "PE")
df_filter = df_filter.repartition(6)


df_filter.explain("formatted")


== Physical Plan ==
AdaptiveSparkPlan (4)
+- Exchange (3)
   +- Filter (2)
      +- Scan csv  (1)


(1) Scan csv 
Output [6]: [id_venta#57, fecha#58, monto#59, pais#60, cliente_id#61, producto#62]
Batched: false
Location: InMemoryFileIndex [file:/C:/Users/User_Kenny/Documents/_KENNEDY_MOLINA_PROYECTOS/pyspark-advanced-databricks/ 02_partitioning_basics/src/ventas.csv]
PushedFilters: [IsNotNull(pais), EqualTo(pais,PE)]
ReadSchema: struct<id_venta:int,fecha:date,monto:double,pais:string,cliente_id:int,producto:string>

(2) Filter
Input [6]: [id_venta#57, fecha#58, monto#59, pais#60, cliente_id#61, producto#62]
Condition : (isnotnull(pais#60) AND (pais#60 = PE))

(3) Exchange
Input [6]: [id_venta#57, fecha#58, monto#59, pais#60, cliente_id#61, producto#62]
Arguments: RoundRobinPartitioning(6), REPARTITION_BY_NUM, [plan_id=60]

(4) AdaptiveSparkPlan
Output [6]: [id_venta#57, fecha#58, monto#59, pais#60, cliente_id#61, producto#62]
Arguments: isFinalPlan=false


